# Vietnamese Hate Speech Detection â€” Fine-tuning PhoBERT on ViHSD

**Task**: 3-class classification of Vietnamese social-media comments â€” `CLEAN` (0), `OFFENSIVE` (1), `HATE` (2).

**Dataset**: [ViHSD](https://www.kaggle.com/datasets/leduc13/vihsd-data) (Luu et al., UIT-NLP) â€” 24,048 train / 2,672 dev / 6,680 test comments, heavily imbalanced (~83% clean, ~7% offensive, ~10% hate).

**Approach**:
1. Baseline: TF-IDF + Logistic Regression.
2. Fine-tune [PhoBERT-base](https://github.com/VinAI/PhoBERT) (VinAI's Vietnamese RoBERTa), with word-segmentation preprocessing (PhoBERT is pretrained on segmented text, so raw syllables would hurt subword tokenization quality) and class-weighted loss to handle the label imbalance.
3. Evaluate with macro-F1 (the right metric here â€” accuracy is misleading under this imbalance) and do qualitative error analysis, especially the OFFENSIVE vs HATE boundary which is the hardest distinction in this dataset.

**Why this project**: content moderation is a real, deployed NLP problem, and Vietnamese is a lower-resource language where naive approaches (raw-syllable tokenization, plain accuracy) visibly fail â€” this notebook shows the fixes and quantifies their effect.

In [ ]:
import os, re, random
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Load data

In [ ]:
DATA_DIR = '/kaggle/input/vihsd-data'
LABEL_NAMES = {0: 'CLEAN', 1: 'OFFENSIVE', 2: 'HATE'}

train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
dev_df   = pd.read_csv(f'{DATA_DIR}/dev.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

for name, df in [('train', train_df), ('dev', dev_df), ('test', test_df)]:
    print(name, df.shape, dict(df['label_id'].value_counts().sort_index()))

train_df.head()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5,3))
counts = train_df['label_id'].value_counts().sort_index()
ax.bar([LABEL_NAMES[i] for i in counts.index], counts.values, color=['#4C72B0','#DD8452','#C44E52'])
ax.set_title('Train label distribution (ViHSD)')
ax.set_ylabel('count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 200, str(v), ha='center')
plt.tight_layout(); plt.show()

## 2. Text cleaning

Light cleaning only â€” strip URLs/mentions/extra whitespace, keep diacritics and punctuation (they carry sentiment/emphasis signal in Vietnamese social text, e.g. repeated punctuation, emoji-adjacent slang).

In [ ]:
URL_RE = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE = re.compile(r'@\w+')
WS_RE = re.compile(r'\s+')

def clean_text(s: str) -> str:
    s = str(s)
    s = URL_RE.sub(' ', s)
    s = MENTION_RE.sub(' ', s)
    s = WS_RE.sub(' ', s).strip()
    return s

for df in (train_df, dev_df, test_df):
    df['text_clean'] = df['free_text'].apply(clean_text)

train_df[['free_text', 'text_clean']].head()

## 3. Baseline â€” TF-IDF + Logistic Regression

Gives a reference point to quantify how much PhoBERT actually buys us, and what a "good enough with 5 minutes of effort" solution looks like.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, confusion_matrix

tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=30000)
Xtr = tfidf.fit_transform(train_df['text_clean'])
Xte = tfidf.transform(test_df['text_clean'])
ytr, yte = train_df['label_id'], test_df['label_id']

clf = LogisticRegression(max_iter=2000, class_weight='balanced', C=2.0)
clf.fit(Xtr, ytr)
pred_baseline = clf.predict(Xte)

baseline_macro_f1 = f1_score(yte, pred_baseline, average='macro')
print('Baseline macro-F1:', round(baseline_macro_f1, 4))
print(classification_report(yte, pred_baseline, target_names=list(LABEL_NAMES.values()), digits=3))

## 4. PhoBERT fine-tuning

PhoBERT's pretraining corpus was word-segmented (multi-syllable Vietnamese words joined with `_`), so we segment with `pyvi` before tokenizing â€” feeding it raw syllables measurably degrades subword quality since the vocabulary expects segmented tokens.

In [ ]:
!pip install -q pyvi

In [ ]:
from pyvi import ViTokenizer

def segment(s: str) -> str:
    try:
        return ViTokenizer.tokenize(s)
    except Exception:
        return s

for df in (train_df, dev_df, test_df):
    df['text_seg'] = df['text_clean'].apply(segment)

print(train_df.loc[0, 'text_clean'])
print(train_df.loc[0, 'text_seg'])

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = 'vinai/phobert-base'
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

class ViHSDDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(list(texts), truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        item['labels'] = self.labels[idx]
        return item

train_ds = ViHSDDataset(train_df['text_seg'], train_df['label_id'])
dev_ds   = ViHSDDataset(dev_df['text_seg'], dev_df['label_id'])
test_ds  = ViHSDDataset(test_df['text_seg'], test_df['label_id'])
len(train_ds), len(dev_ds), len(test_ds)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1, 2])
class_weights = compute_class_weight('balanced', classes=classes, y=train_df['label_id'].values)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print('class weights:', class_weights)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(DEVICE)

In [ ]:
from transformers import Trainer, TrainingArguments
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': (preds == labels).mean(),
        'macro_f1': f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
    }

args = TrainingArguments(
    output_dir='/kaggle/working/phobert-vihsd',
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to='none',
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

## 5. Test set evaluation

In [ ]:
test_output = trainer.predict(test_ds)
test_logits, test_labels = test_output.predictions, test_output.label_ids
test_preds = np.argmax(test_logits, axis=-1)

phobert_macro_f1 = f1_score(test_labels, test_preds, average='macro')
print('PhoBERT macro-F1 (test):', round(phobert_macro_f1, 4))
print('Baseline macro-F1 (test):', round(baseline_macro_f1, 4))
print(classification_report(test_labels, test_preds, target_names=list(LABEL_NAMES.values()), digits=3))

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(4,4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3)); ax.set_xticklabels(LABEL_NAMES.values())
ax.set_yticks(range(3)); ax.set_yticklabels(LABEL_NAMES.values())
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black')
ax.set_title('PhoBERT confusion matrix (test)')
plt.tight_layout(); plt.show()

## 6. Error analysis

Focus on the hardest boundary: OFFENSIVE vs HATE. These differ mainly by *target and intent* (general profanity vs. targeted attack on a group/individual), which is exactly the kind of pragmatic distinction that surface-level features struggle with.

In [ ]:
errors = test_df.copy()
errors['pred'] = test_preds
errors['true'] = test_labels
mis = errors[errors['pred'] != errors['true']]

off_hate_confusions = mis[mis['true'].isin([1,2]) & mis['pred'].isin([1,2])]
print(f'{len(mis)} total misclassifications on test set ({len(mis)/len(errors):.1%})')
print(f'{len(off_hate_confusions)} of which are OFFENSIVE<->HATE confusions')

pd.set_option('display.max_colwidth', 120)
off_hate_confusions[['free_text','true','pred']].assign(
    true=lambda d: d['true'].map(LABEL_NAMES), pred=lambda d: d['pred'].map(LABEL_NAMES)
).head(15)

## 7. Save model

In [ ]:
SAVE_DIR = '/kaggle/working/phobert-vihsd-final'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print('saved to', SAVE_DIR)

## 8. Summary

| Model | Macro-F1 (test) |
|---|---|
| TF-IDF + Logistic Regression | see cell 3 output above |
| PhoBERT-base (fine-tuned, word-segmented, class-weighted) | see cell 5 output above |

**Limitations / future work**:
- OFFENSIVE vs HATE remains the dominant error mode â€” could be improved with more context (thread-level, not single-comment) or a larger/augmented HATE sample (it's the rarest class).
- No adversarial/teencode-normalization step (e.g. `mk` â†’ `mÃ y`, elongated letters) â€” worth testing whether a slang-normalization pass before segmentation helps.
- Single random seed reported; for a rigorous CV bullet, rerun with 3 seeds and report mean Â± std.